In [1]:
import importlib
import os
from pathlib import Path

import pandas as pd

import mp_stable_phases
importlib.reload(mp_stable_phases)

from mp_stable_phases import fetch_stable_phases_for_elements, save_stable_phases_table

from build_datasheet import (
    build_datasheet,
    elemental_props_vs_host,
    optimized_dopant_list,
    save_datasheet,
)
from mp_structure_features import (
    fetch_and_featurize_phases,
    featurize_phases_with_prefix,
    save_structure_features,
)
from endmember_phases import (
    build_endmember_table,
    default_endmember_matminer_path,
    default_endmember_table_path,
    fetch_endmember_mp_metadata,
    save_endmember_table,
)


In [2]:
ML_DIR = Path(".").resolve()
DATA_DIR = ML_DIR / "data"
REPO_ROOT = ML_DIR.parent
DATA_DIR.mkdir(exist_ok=True)


In [3]:
MP_API_KEY = os.environ.get("MP_API_KEY")


In [4]:
from dataclasses import dataclass

@dataclass
class ElementData:
    Z: int               # atomic number
    r_atomic: float      # pm
    crystal: str         # polytype
    valency: int         # unit 
    chi: float           # Pauling scale


# Complete self-contained data for all elements (atomic number, metallic radius from XRD, crystal structure, valency, Pauling electronegativity χ)
ELEMENTS_DATA = {
    "H":  ElementData(1,   31, "molecular",  1,  2.20),
    "Li": ElementData(3,  152, "bcc",        1,  0.98),
    "Be": ElementData(4,  112, "hcp",        2,  1.57),
    "B":  ElementData(5,   85, "beta_rhombohedral", 3,  2.04), 
    "Na": ElementData(11,  186, "bcc",        1,  0.93),
    "Mg": ElementData(12,  160, "hcp",        2,  1.31),
    "Al": ElementData(13,  143, "fcc",        3,  1.61),
    "Si": ElementData(14,  111, "diamond",    4,  1.90),
    "K":  ElementData(19,  227, "bcc",        1,  0.82),
    "Ca": ElementData(20,  197, "fcc",        2,  1.00),
    "Sc": ElementData(21,  162, "hcp",        3,  1.36),
    "Ti": ElementData(22,  147, "hcp",        4,  1.54),
    "V":  ElementData(23,  134, "bcc",        5,  1.63),
    "Cr": ElementData(24,  128, "bcc",        3,  1.66),  # Cr  +3, +6
    "Mn": ElementData(25,  127, "complex",    2,  1.55),  # Mn  +2
    "Fe": ElementData(26,  126, "bcc",        2,  1.83),  # Fe  +2, +3
    "Co": ElementData(27,  125, "hcp",        2,  1.88),  # Co  +2
    "Ni": ElementData(28,  124, "fcc",        2,  1.91),  # Ni  +2
    "Cu": ElementData(29,  128, "fcc",        1,  1.90),  # Cu +1
    "Zn": ElementData(30,  134, "hcp",        2,  1.65),  # Zn +2
    "Ga": ElementData(31,  135, "ortho",      3,  1.81),
    "Ge": ElementData(32,  122, "diamond",    4,  2.01),
    "As": ElementData(33,  119, "rhombo",     5,  2.18),
    "Rb": ElementData(37,  248, "bcc",        1,  0.82),
    "Sr": ElementData(38,  215, "fcc",        2,  0.95),
    "Y":  ElementData(39,  180, "hcp",        3,  1.22),
    "Zr": ElementData(40,  160, "hcp",        4,  1.33),
    "Nb": ElementData(41,  146, "bcc",        5,  1.60),
    "Mo": ElementData(42,  139, "bcc",        6,  2.16),  # Mo  +6
    "Tc": ElementData(43,  136, "hcp",        7,  1.90),
    "Ru": ElementData(44,  134, "hcp",        3,  2.20),  # Ru  +3
    "Rh": ElementData(45,  134, "fcc",        3,  2.28),  # Rh  +3
    "Pd": ElementData(46,  137, "fcc",        2,  2.20),  # Pd  +2
    "Ag": ElementData(47,  144, "fcc",        1,  1.93),  # Ag +1
    "Cd": ElementData(48,  151, "hcp",        2,  1.69),  # Cd +2
    "In": ElementData(49,  167, "tetra",      3,  1.78),
    "Sn": ElementData(50,  140, "tetra",      4,  1.96),
    "Sb": ElementData(51,  140, "rhombo",     5,  2.05),
    "Cs": ElementData(55,  265, "bcc",        1,  0.79),
    "Ba": ElementData(56,  222, "bcc",        2,  0.89),
    "Hf": ElementData(72,  159, "hcp",        4,  1.30),
    "Ta": ElementData(73,  146, "bcc",        5,  1.50),
    "W":  ElementData(74,  139, "bcc",        6,  2.36),  # W  +6
    "Re": ElementData(75,  137, "hcp",        7,  1.90),
    "Os": ElementData(76,  135, "hcp",        4,  2.20),  # Os  +4
    "Ir": ElementData(77,  136, "fcc",        3,  2.20),  # Ir  +3
    "Pt": ElementData(78,  139, "fcc",        2,  2.28),  # Pt  +2
    "Au": ElementData(79,  144, "fcc",        1,  2.54),  # Au +1
    "Hg": ElementData(80,  151, "rhombo",     2,  2.00),  # Hg +2
    "Bi": ElementData(83,  155, "rhombo",     5,  2.02),
    "La": ElementData(57,  187, "dhcp",       3,  1.10),
    "Ce": ElementData(58,  182, "fcc",        3,  1.12),
    "Pr": ElementData(59,  182, "hcp",        3,  1.13),
    "Nd": ElementData(60,  181, "hcp",        3,  1.14),
    "Sm": ElementData(62,  180, "rhombo",     3,  1.17),
    "Eu": ElementData(63,  208, "bcc",        2,  0.63),  # Eu  +2
    "Gd": ElementData(64,  180, "hcp",        3,  1.20),
    "Tb": ElementData(65,  177, "hcp",        3,  1.22),
    "Dy": ElementData(66,  178, "hcp",        3,  1.22),
    "Ho": ElementData(67,  176, "hcp",        3,  1.23),
    "Er": ElementData(68,  176, "hcp",        3,  1.24),
    "Tm": ElementData(69,  176, "hcp",        3,  1.25),
    "Yb": ElementData(70,  194, "fcc",        2,  1.10),  # Yb  +2
    "Lu": ElementData(71,  174, "hcp",        3,  1.27),

    "Se": ElementData(34, 116, "trigonal",     6,  2.55),
    "Tl": ElementData(81, 170, "hcp",          3,  1.62),
    "Pb": ElementData(82, 175, "fcc",          4,  2.33),
}

def get_element_data(sym: str) -> ElementData:
    sym = sym.strip().title()
    data = ELEMENTS_DATA.get(sym)
    if data is None:
        raise ValueError(f"Element '{sym}' not found in ELEMENTS_DATA")
    return data


def element_radii_lookup() -> dict[str, float]:
    """Radii (pm) from ELEMENTS_DATA for MP/matminer helpers."""
    return {sym: float(data.r_atomic) for sym, data in ELEMENTS_DATA.items()}


In [5]:
host_element = 'Li'


In [6]:
# Only dopants with optimized vacancy energies (binding + solution)
dopant_list = optimized_dopant_list(REPO_ROOT)
print(f"{len(dopant_list)} optimized dopants: {dopant_list}")

elemental_props_df = elemental_props_vs_host(
    dopant_list,
    host_element,
    ELEMENTS_DATA,
    get_element_data=get_element_data,
)
elemental_props_df.to_csv(DATA_DIR / "elemental_props_vs_li.csv", index=False)
print(f"Saved elemental properties vs Li: {DATA_DIR / 'elemental_props_vs_li.csv'}")
elemental_props_df


19 optimized dopants: ['Ag', 'Au', 'Bi', 'Cd', 'Cu', 'Ga', 'Ge', 'Hg', 'In', 'Mg', 'Na', 'Pb', 'Pd', 'Pt', 'Rh', 'Sb', 'Sn', 'Tl', 'Zn']
Saved elemental properties vs Li: /home/a.burov/li_alloys/ml_part/data/elemental_props_vs_li.csv


,element,delta_r_vs_Li_pm,valency,delta_valency_vs_Li,chi,delta_chi_vs_Li
0,Ag,-8,1,0,1.93,0.95
1,Au,-8,1,0,2.54,1.56
2,Bi,3,5,4,2.02,1.04
3,Cd,-1,2,1,1.69,0.71
4,Cu,-24,1,0,1.90,0.92
5,Ga,-17,3,2,1.81,0.83
6,Ge,-30,4,3,2.01,1.03
7,Hg,-1,2,1,2.00,1.02
8,In,15,3,2,1.78,0.80
9,Mg,8,2,1,1.31,0.33


In [7]:
# Fetch stable MP phases (no absolute radius columns in output)
stable_phases_df = fetch_stable_phases_for_elements(
    dopant_list,
    host_element=host_element,
    api_key=MP_API_KEY,
    e_hull_max=0.02,
    strict_less_than=True,
    selection_mode="min_nsites",
    require_experimental=True,
    prefer_is_stable=False,
    use_rt_mp_catalog=False,
    allow_theoretical_fallback=False,
    include_manual_radii=False,
)

save_stable_phases_table(
    stable_phases_df,
    DATA_DIR / "dopant_stable_experimental_phases_mp.csv",
)
stable_phases_df


Retrieving SummaryDoc documents:   0%|          | 0/9 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/5 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/4 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/17 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/5 [00:00<?, ?it/s]

...

Retrieving SummaryDoc documents:   0%|          | 0/14 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/9 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/6 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/8 [00:00<?, ?it/s]

,element,role,material_id,formula_pretty,energy_per_atom,energy_above_hull,theoretical,is_stable,crystal_system,spacegroup_symbol,...,volume,nsites,selection_rule,e_hull_max_eV_per_atom,is_theoretical_fallback,n_candidates,n_experimental,n_near_hull,material_id_forced,error
0,Li,host,mp-51,Li,-2.385034,0.001672,False,False,Cubic,Fm-3m,...,20.347259,1,hull_window_min_nsites_lt,0.02,False,9,5,5,False,None
1,Ag,dopant,mp-124,Ag,-21.353750,0.002127,False,False,Cubic,Fm-3m,...,17.285231,1,hull_window_min_nsites_lt,0.02,False,5,3,3,False,None
2,Au,dopant,mp-81,Au,-50.584960,0.000000,False,True,Cubic,Fm-3m,...,18.144738,1,hull_window_min_nsites_lt,0.02,False,4,1,1,False,None
3,Bi,dopant,mp-23152,Bi,-58.502274,0.000000,False,True,Trigonal,R-3m,...,73.171978,2,hull_window_min_nsites_lt,0.02,False,17,10,1,False,None
4,Cd,dopant,mp-94,Cd,-20.104988,0.000000,False,True,Hexagonal,P6_3/mmc,...,44.615039,2,hull_window_min_nsites_lt,0.02,False,5,1,1,False,None
5,Cu,dopant,mp-30,Cu,-10.846796,0.000000,False,True,Cubic,Fm-3m,...,11.445999,1,hull_window_min_nsites_lt,0.02,False,8,2,1,False,None
6,Ga,dopant,mp-142,Ga,-11.442776,0.000000,False,True,Orthorhombic,Cmce,...,76.702421,4,hull_window_min_nsites_lt,0.02,False,8,6,1,False,None
7,Ge,dopant,mp-32,Ge,-13.871978,0.000000,False,True,Cubic,Fd-3m,...,45.688209,2,hull_window_min_nsites_lt,0.02,False,17,14,3,False,None
8,Hg,dopant,mp-10861,Hg,-49.364846,0.002960,False,False,Hexagonal,P6/mmm,...,82.436519,3,forced_mp_id,0.02,False,1,1,1,True,None
9,In,dopant,mp-85,In,-22.578590,0.000000,False,True,Cubic,Fm-3m,...,26.392515,1,hull_window_min_nsites_lt,0.02,False,7,2,2,False,None


In [8]:
phase_structures, matminer_features_df = fetch_and_featurize_phases(
    stable_phases_df,
    api_key=MP_API_KEY,
)

save_structure_features(
    matminer_features_df,
    DATA_DIR / "structure_matminer_features.csv",
)
matminer_features_df.head()


Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

...

/home/a.burov/micromamba/envs/msdb/lib/python3.10/site-packages/matminer/utils/data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


ElementProperty:   0%|          | 0/20 [00:00<?, ?it/s]

DensityFeatures:   0%|          | 0/20 [00:00<?, ?it/s]

GlobalSymmetryFeatures:   0%|          | 0/20 [00:00<?, ?it/s]

,element,material_id,role,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,...,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber,density,vpa,packing fraction,spacegroup_num,crystal_system,crystal_system_int,is_centrosymmetric,n_symmetry_ops
0,Li,mp-51,host,3.0,3.0,0.0,3.0,0.0,3.0,1.0,...,0.0,229.0,0.566455,20.347259,0.627605,225,cubic,1,True,48
1,Ag,mp-124,dopant,47.0,47.0,0.0,47.0,0.0,47.0,65.0,...,0.0,225.0,10.362567,17.285231,0.992598,225,cubic,1,True,48
2,Au,mp-81,dopant,79.0,79.0,0.0,79.0,0.0,79.0,66.0,...,0.0,225.0,18.025649,18.144738,0.567988,225,cubic,1,True,48
3,Bi,mp-23152,dopant,83.0,83.0,0.0,83.0,0.0,83.0,86.0,...,0.0,12.0,9.485055,36.585989,0.468958,166,trigonal,3,True,12
4,Cd,mp-94,dopant,48.0,48.0,0.0,48.0,0.0,48.0,70.0,...,0.0,194.0,8.367710,22.307519,0.699250,194,hexagonal,2,True,24


In [9]:
# Li–X supercell endmember reference phases (non-Li species from end_members_dir)
endmember_table = build_endmember_table()
endmember_phases_df = fetch_endmember_mp_metadata(endmember_table, api_key=MP_API_KEY)
endmember_phases_path = save_endmember_table(
    endmember_phases_df,
    default_endmember_table_path(ML_DIR),
)
print(f"Saved {len(endmember_phases_df)} endmember rows to {endmember_phases_path}")
endmember_phases_df


Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

...

Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Saved 50 endmember rows to /home/a.burov/li_alloys/ml_part/data/dopant_endmember_phases_mp.csv


,element,endmember_formula,endmember_material_id_raw,endmember_material_id,endmember_fraction,endmember_mp_material_id,endmember_formula_pretty,endmember_energy_per_atom,endmember_energy_above_hull,endmember_theoretical,...,endmember_volume,endmember_nsites,endmember_selection_rule,endmember_e_hull_max_eV_per_atom,endmember_is_theoretical_fallback,endmember_n_candidates,endmember_n_experimental,endmember_n_near_hull,endmember_material_id_forced,endmember_error
0,Ag,LiAg,mp-2426-r2SCAN,mp-2426,0.015625,mp-2426,LiAg,-12.051744,0.000000,False,...,30.451425,2,endmember_reference,0.1,False,1,1,1,True,None
1,Al,Li9Al4,mp-568404-r2SCAN,mp-568404,0.025391,mp-568404,Li9Al4,-3.892354,0.000000,False,...,214.505258,13,endmember_reference,0.1,False,1,1,1,True,None
2,Au,Li15Au4,mp-567395-r2SCAN,mp-567395,0.037109,mp-567395,Li15Au4,-12.887391,0.000000,False,...,612.719827,38,endmember_reference,0.1,False,1,1,1,True,None
3,B,LiB,mp-1001835-r2SCAN,mp-1001835,0.015625,mp-1001835,LiB,-5.091218,0.000000,False,...,43.101252,4,endmember_reference,0.1,False,1,1,1,True,None
4,Be,Be,mp-87-GGA,mp-87,0.007812,mp-87,Be,-3.739413,0.000000,False,...,31.796370,4,endmember_reference,0.1,False,1,1,1,True,None
5,Bi,Li3Bi,mp-23222-r2SCAN,mp-23222,0.031250,mp-23222,Li3Bi,-17.043428,0.000000,False,...,74.887894,4,endmember_reference,0.1,False,1,1,1,True,None
6,Ca,Li2Ca,mp-570466-r2SCAN,mp-570466,0.007812,mp-570466,Li2Ca,-3.956578,0.000000,False,...,338.280424,12,endmember_reference,0.1,False,1,1,1,True,None
7,Cd,LiCd,mp-1437-r2SCAN,mp-1437,0.015625,mp-1437,LiCd,-11.521597,0.000000,False,...,74.013228,4,endmember_reference,0.1,False,1,1,1,True,None
8,Ce,Ce,mp-28-r2SCAN,mp-28,0.007812,mp-28,Ce,-30.840629,0.000000,False,...,25.501645,1,endmember_reference,0.1,False,1,1,1,True,None
9,Co,Co,mp-102-r2SCAN,mp-102,0.007812,mp-102,Co,-13.234231,0.000000,False,...,10.841543,1,endmember_reference,0.1,False,1,1,1,True,None


In [10]:
# Matminer structure features for endmember phases (endmember_* column prefix)
endmember_featurize_df = endmember_table.dropna(subset=["endmember_material_id"]).copy()
endmember_featurize_df = endmember_featurize_df.rename(
    columns={"endmember_material_id": "material_id"}
)
endmember_featurize_df["role"] = "endmember"
_, endmember_matminer_df = featurize_phases_with_prefix(
    endmember_featurize_df[["element", "material_id", "role", "endmember_formula"]],
    api_key=MP_API_KEY,
    prefix="endmember_",
    extra_id_cols=("endmember_formula",),
)
endmember_matminer_path = save_structure_features(
    endmember_matminer_df,
    default_endmember_matminer_path(ML_DIR),
)
print(f"Saved endmember matminer features to {endmember_matminer_path}")
endmember_matminer_df.head()


Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

...

ElementProperty:   0%|          | 0/50 [00:00<?, ?it/s]

DensityFeatures:   0%|          | 0/50 [00:00<?, ?it/s]

GlobalSymmetryFeatures:   0%|          | 0/50 [00:00<?, ?it/s]

Saved endmember matminer features to /home/a.burov/li_alloys/ml_part/data/structure_endmember_matminer_features.csv


,element,endmember_material_id,role,endmember_MagpieData minimum Number,endmember_MagpieData maximum Number,endmember_MagpieData range Number,endmember_MagpieData mean Number,endmember_MagpieData avg_dev Number,endmember_MagpieData mode Number,endmember_MagpieData minimum MendeleevNumber,...,endmember_MagpieData avg_dev SpaceGroupNumber,endmember_MagpieData mode SpaceGroupNumber,endmember_density,endmember_vpa,endmember_packing fraction,endmember_spacegroup_num,endmember_crystal_system,endmember_crystal_system_int,endmember_is_centrosymmetric,endmember_n_symmetry_ops
0,Ag,mp-2426,endmember,3.0,47.0,44.0,25.000000,22.000000,3.0,1.0,...,2.000000,225.0,6.260632,15.225712,0.982789,221,cubic,1,True,48
1,Al,mp-568404,endmember,3.0,13.0,10.0,6.076923,4.260355,3.0,1.0,...,1.704142,229.0,1.319072,16.500404,0.688353,2,triclinic,7,True,2
2,Au,mp-567395,endmember,3.0,79.0,76.0,19.000000,25.263158,3.0,1.0,...,1.329640,229.0,4.834737,16.124206,0.759808,220,cubic,1,False,24
3,B,mp-1001835,endmember,3.0,5.0,2.0,4.000000,1.000000,3.0,1.0,...,31.500000,166.0,1.367844,10.775313,0.711928,194,hexagonal,2,True,24
4,Be,mp-87,endmember,4.0,4.0,0.0,4.000000,0.000000,4.0,67.0,...,0.000000,194.0,1.882615,7.949093,0.610013,194,hexagonal,2,True,16


In [11]:
datasheet_df = build_datasheet(
    REPO_ROOT,
    mp_phases_path=DATA_DIR / "dopant_stable_experimental_phases_mp.csv",
    elemental_props_df=elemental_props_df,
)

datasheet_path = save_datasheet(datasheet_df, DATA_DIR)
print(f"Saved: {datasheet_path} ({len(datasheet_df)} dopants)")
datasheet_df.head()


Saved: /home/a.burov/li_alloys/ml_part/data/datasheet.csv (19 dopants)


,element,vac_binding_energy_eV,vac_position,solution_energy_eV,mp_material_id,mp_formula_pretty,mp_energy_per_atom,mp_energy_above_hull,mp_theoretical,mp_is_stable,...,endmember_volume,endmember_nsites,endmember_selection_rule,endmember_e_hull_max_eV_per_atom,endmember_is_theoretical_fallback,endmember_n_candidates,endmember_n_experimental,endmember_n_near_hull,endmember_material_id_forced,endmember_error
0,Na,0.07742,near,0.18167,mp-127,Na,-3.548408,0.015772,0,0,...,37.257507,1,endmember_reference,0.1,False,1,1,1,True,NaN
1,Mg,0.00961,2NN,-0.15375,mp-153,Mg,-4.169304,0.000000,0,1,...,44.798996,2,endmember_reference,0.1,False,1,1,1,True,NaN
2,Cu,0.06406,2NN,0.13912,mp-30,Cu,-10.846796,0.000000,0,1,...,11.445999,1,endmember_reference,0.1,False,1,1,1,True,NaN
3,Zn,0.07767,near,0.07941,mp-79,Zn,-8.910603,0.000000,0,1,...,55.736715,4,endmember_reference,0.1,False,1,1,1,True,NaN
4,Ga,0.09961,near,0.06493,mp-142,Ga,-11.442776,0.000000,0,1,...,90.331803,6,endmember_reference,0.1,False,1,1,1,True,NaN


In [12]:
datasheet_df.head(30)


,element,vac_binding_energy_eV,vac_position,solution_energy_eV,mp_material_id,mp_formula_pretty,mp_energy_per_atom,mp_energy_above_hull,mp_theoretical,mp_is_stable,...,endmember_volume,endmember_nsites,endmember_selection_rule,endmember_e_hull_max_eV_per_atom,endmember_is_theoretical_fallback,endmember_n_candidates,endmember_n_experimental,endmember_n_near_hull,endmember_material_id_forced,endmember_error
0,Na,0.07742,near,0.18167,mp-127,Na,-3.548408,0.015772,0,0,...,37.257507,1,endmember_reference,0.1,False,1,1,1,True,NaN
1,Mg,0.00961,2NN,-0.15375,mp-153,Mg,-4.169304,0.000000,0,1,...,44.798996,2,endmember_reference,0.1,False,1,1,1,True,NaN
2,Cu,0.06406,2NN,0.13912,mp-30,Cu,-10.846796,0.000000,0,1,...,11.445999,1,endmember_reference,0.1,False,1,1,1,True,NaN
3,Zn,0.07767,near,0.07941,mp-79,Zn,-8.910603,0.000000,0,1,...,55.736715,4,endmember_reference,0.1,False,1,1,1,True,NaN
4,Ga,0.09961,near,0.06493,mp-142,Ga,-11.442776,0.000000,0,1,...,90.331803,6,endmember_reference,0.1,False,1,1,1,True,NaN
5,Ge,0.22611,near,0.21988,mp-32,Ge,-13.871978,0.000000,0,1,...,594.489591,38,endmember_reference,0.1,False,1,1,1,True,NaN
6,Rh,0.09293,2NN,0.04169,mp-74,Rh,-24.136229,0.000000,0,1,...,26.358645,2,endmember_reference,0.1,False,1,1,1,True,NaN
7,Pd,0.14238,near,0.16735,mp-2,Pd,-22.960454,0.000000,0,1,...,588.565526,38,endmember_reference,0.1,False,1,1,1,True,NaN
8,Ag,0.12930,near,-0.11688,mp-124,Ag,-21.353750,0.002127,0,0,...,30.451425,2,endmember_reference,0.1,False,1,1,1,True,NaN
9,Cd,0.07245,2NN,-0.21207,mp-94,Cd,-20.104988,0.000000,0,1,...,74.013228,4,endmember_reference,0.1,False,1,1,1,True,NaN
